# Day 35: Monitoring Reliability Improvements

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Protect local event writes and add a clearly scoped prediction-distribution comparison.

## Theory

Concurrent CSV writes can lose events or corrupt rows. A process-local lock serializes those operations. Changes in predicted class frequencies describe usage shifts, not verified performance degradation.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Exercise concurrency, existing monitoring and privacy regression tests.

In [2]:
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_monitoring.py","tests/test_monitoring_reliability.py","tests/test_security.py"],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0,r.stderr

................                                                         [100%]
============================== warnings summary ===============================
..\venv\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
16 passed, 1 warning in 2.73s



Compare outputs for two explicitly defined public-data windows.

In [3]:
from src.dataset import original_split
from src.prediction import predict
from src.monitoring import compare_prediction_distribution
X=original_split()[1]
a=predict(X.iloc[:65]).predicted_category
b=predict(X.iloc[65:]).predicted_category
report=compare_prediction_distribution(a,b)
print(json.dumps(report,indent=2))
Path("reports/monitoring_distribution.json").write_text(json.dumps(report,indent=2))

{
  "reference_count": 65,
  "current_count": 65,
  "total_variation": 0.06153846153846154,
  "reference_proportions": {
    "Low": 0.18461538461538463,
    "Medium": 0.6153846153846154,
    "High": 0.2
  },
  "current_proportions": {
    "Low": 0.24615384615384617,
    "Medium": 0.5846153846153846,
    "High": 0.16923076923076924
  },
  "interpretation": "Descriptive output shift; does not establish model degradation or accuracy."
}


437

## Findings and interpretation

The concurrency test writes 40 isolated test events and verifies every event survives. The public-data window comparison records a descriptive shift without attaching an accuracy claim.

## Limitations

The CSV lock does not protect multiple processes or cloud replicas. The two windows are historical demo records, not production traffic.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Protect local event writes and add a clearly scoped prediction-distribution comparison.

## Next-day plan

Record data and model versions with hashes and provenance.